In [ ]:
# ==========================================
# 1. IMPORT LIBRARY & LOAD DATASET
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, GRU, Dense, Dropout
from sklearn.metrics import classification_report, confusion_matrix

# Parameter konfigurasi
MAX_WORDS = 10000  # Hanya gunakan 10.000 kata paling sering muncul
MAX_LEN = 200      # Panjang maksimal kata per review
EMBEDDING_DIM = 64

print("Memuat dataset IMDB...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=MAX_WORDS)

# Padding agar seluruh teks memiliki panjang sekuens yang sama (200)
x_train = pad_sequences(x_train, maxlen=MAX_LEN, padding='post', truncating='post')
x_test = pad_sequences(x_test, maxlen=MAX_LEN, padding='post', truncating='post')

print(f"Shape Data Train : {x_train.shape}")
print(f"Shape Data Test  : {x_test.shape}")


In [ ]:
# ==========================================
# 2. ARSITEKTUR 1: VANILLA RNN (SimpleRNN)
# ==========================================
model_rnn = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    SimpleRNN(64, return_sequences=False),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
], name="Vanilla_RNN")

model_rnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_rnn.summary()

print("\nMelatih Model SimpleRNN...")
history_rnn = model_rnn.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)


In [ ]:
# ==========================================
# 3. ARSITEKTUR 2: GRU (Gated Recurrent Unit)
# ==========================================
model_gru = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    GRU(64, return_sequences=False),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
], name="GRU_Model")

model_gru.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_gru.summary()

print("\nMelatih Model GRU...")
history_gru = model_gru.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)


In [ ]:
# ==========================================
# 4. VISUALISASI & PERBANDINGAN PERFORMA
# ==========================================
def plot_comparison(h1, h2, title1="SimpleRNN", title2="GRU"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot Akurasi Validasi
    axes[0].plot(h1.history['val_accuracy'], label=f'{title1} Val Acc', marker='o')
    axes[0].plot(h2.history['val_accuracy'], label=f'{title2} Val Acc', marker='s')
    axes[0].set_title('Perbandingan Validation Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)

    # Plot Loss Validasi
    axes[1].plot(h1.history['val_loss'], label=f'{title1} Val Loss', marker='o')
    axes[1].plot(h2.history['val_loss'], label=f'{title2} Val Loss', marker='s')
    axes[1].set_title('Perbandingan Validation Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

plot_comparison(history_rnn, history_gru)


In [ ]:
# ==========================================
# 5. EVALUASI PADA DATA TEST
# ==========================================
for name, model in [("SimpleRNN", model_rnn), ("GRU", model_gru)]:
    print(f"\n{'='*40}\nEvaluasi Model: {name}\n{'='*40}")
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"Test Accuracy: {acc*100:.2f}% | Test Loss: {loss:.4f}\n")

    y_pred = (model.predict(x_test, verbose=0) > 0.5).astype("int32")
    print(classification_report(y_test, y_pred, target_names=['Negatif (0)', 'Positif (1)']))
